# 선형회귀 모델 작성, 예측, 평가

In [1]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
%matplotlib inline

In [2]:
charges_df = pd.read_csv('./머신러닝 보고서/data1/premium.csv')
charges_df.head()

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


In [18]:
# 'bmi' 컬럼 결측값에 평균값으로 채우기
charges_df['bmi'] = charges_df['bmi'].fillna(charges_df['bmi'].mean())

+ 'sex', 'smoker', 'region' 컬럼 레이블 인코딩

In [19]:
from sklearn.preprocessing import LabelEncoder

# 레이블 인코딩을 적용할 컬럼 리스트
columns_label = ['sex', 'smoker', 'region']

# 복사본 생성
charges_df1 = charges_df.copy()

# 각 컬럼에 대해 LabelEncoder 적용
for col in columns_label:
    if col in charges_df1.columns: # 컬럼 존재 여부 확인 (방어 코드)
        le = LabelEncoder() # 각 컬럼마다 새로운 LabelEncoder 객체를 생성
        charges_df1[col] = le.fit_transform(charges_df1[col])

        print(f"\n--- '{col}' 컬럼 인코딩 후 매핑 정보 ---")
        # 인코딩된 값과 원래 문자열 값의 매핑 확인
        for i, class_name in enumerate(le.classes_):
            print(f"  {class_name}: {i}")
    else:
        print(f"경고: '{col}' 컬럼이 데이터프레임에 존재하지 않습니다.")

print("\n--- 레이블 인코딩 후 데이터 head() ---")
print(charges_df1.head())

print("\n--- 레이블 인코딩 후 데이터 컬럼 Dtypes 확인 ---")
print(charges_df1.dtypes)


--- 'sex' 컬럼 인코딩 후 매핑 정보 ---
  female: 0
  male: 1

--- 'smoker' 컬럼 인코딩 후 매핑 정보 ---
  no: 0
  yes: 1

--- 'region' 컬럼 인코딩 후 매핑 정보 ---
  northeast: 0
  northwest: 1
  southeast: 2
  southwest: 3

--- 레이블 인코딩 후 데이터 head() ---
   age  sex     bmi  children  smoker  region      charges
0   19    0  27.900         0       1       3  16884.92400
1   18    1  33.770         1       0       2   1725.55230
2   28    1  33.000         3       0       2   4449.46200
3   33    1  22.705         0       0       1  21984.47061
4   32    1  28.880         0       0       1   3866.85520

--- 레이블 인코딩 후 데이터 컬럼 Dtypes 확인 ---
age           int64
sex           int64
bmi         float64
children      int64
smoker        int64
region        int64
charges     float64
dtype: object


In [20]:
X = charges_df1.drop('charges', axis=1).values   # 독립변수
y = charges_df1['charges'].values   # 종속변수

## 모델 만들기

In [21]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=156)
lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred = lr.predict(X_test)
y_pred[:3]

array([14474.70246359, -1367.94174448, 11182.0795591 ])

In [23]:
# 평가
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
rmse, mse

(np.float64(5892.287122021437), np.float64(34719047.52833967))

+ RMSE는 약 5892.29이다

In [24]:
# 결정계수
r2_score(y_test, y_pred)

np.float64(0.7314050294401666)

+ 결정계수 r2스코어는 약 0.731이다

In [26]:
# 회귀식
# w0, w1
lr.intercept_, lr.coef_

(np.float64(-12749.427561095086),
 array([  257.5385329 ,  -339.97677884,   369.66261678,   471.40493778,
        23624.46109983,  -375.59873801]))

In [27]:
np.round(lr.intercept_, 1), np.round(lr.coef_, 1)

(np.float64(-12749.4),
 array([  257.5,  -340. ,   369.7,   471.4, 23624.5,  -375.6]))

In [29]:
pd.Series(data=np.round(lr.coef_, 1), index=charges_df1.drop('charges', axis=1).columns).sort_values(ascending=False)

smoker      23624.5
children      471.4
bmi           369.7
age           257.5
sex          -340.0
region       -375.6
dtype: float64

# 랜덤포레스트회귀 모델 작성, 예측, 평가

In [30]:
from sklearn.ensemble import RandomForestRegressor

In [31]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=156)
rf = RandomForestRegressor()
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)
y_pred[:3]

array([15533.1786617,  1654.904393 ,  9662.7078957])

In [32]:
# 평가
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
rmse, mse

(np.float64(4810.35507765473), np.float64(23139515.973118644))

+ RMSE는 약 4810.36이다

In [33]:
# 결정계수
r2_score(y_test, y_pred)

np.float64(0.8209870934248577)

+ 결정계수 r2스코어는 약 0.821이다

In [42]:
# 피처 중요도 확인
feature_importances = rf.feature_importances_
feature_names = charges_df1.drop('charges', axis=1).columns

# 각 피처의 이름과 중요도를 매칭하여 출력
for feature, importance in zip(feature_names, feature_importances):
    print(f"{feature}: {importance:.4f}")

age: 0.1231
sex: 0.0055
bmi: 0.2132
children: 0.0181
smoker: 0.6245
region: 0.0156


## 교차 검증

In [45]:
# 선형회귀모델 교차 검증
from sklearn.model_selection import cross_val_score
neg_mse_score = cross_val_score(lr, X, y, scoring='neg_mean_squared_error', cv=5)
neg_mse_score

array([-37353966.14780101, -38018280.71475136, -32981193.39000173,
       -39560881.14778336, -37174240.90789752])

In [46]:
# MSE, RMSE
RMSE = np.sqrt(neg_mse_score * -1)
np.mean(RMSE), RMSE

(np.float64(6081.484710559383),
 array([6111.78911186, 6165.89658645, 5742.92550796, 6289.74412419,
        6097.06822234]))

+ 교차검증 후 RMSE는 약 6081.48이다

In [47]:
# R2
r2_scores = cross_val_score(lr, X, y, scoring='r2', cv=5)
r2_scores, np.mean(r2_scores)

(array([0.75962321, 0.70729102, 0.77528105, 0.73350581, 0.7552539 ]),
 np.float64(0.7461909971637163))

+ 교차검증 후 R2스코어는 약 0.75이다

In [48]:
# 랜덤포레스트모델 교차 검증
from sklearn.model_selection import cross_val_score
neg_mse_score = cross_val_score(rf, X, y, scoring='neg_mean_squared_error', cv=5)
neg_mse_score

array([-23029214.27132077, -29720813.26495486, -22483887.37831673,
       -25506182.12662895, -22226039.60775219])

In [49]:
# MSE, RMSE
RMSE = np.sqrt(neg_mse_score * -1)
np.mean(RMSE), RMSE

(np.float64(4951.417723619518),
 array([4798.87635508, 5451.67985716, 4741.71776662, 5050.36455383,
        4714.4500854 ]))

+ 교차검증 후 RMSE는 약 4951.42이다

In [51]:
# R2
r2_scores = cross_val_score(rf, X, y, scoring='r2', cv=5)
r2_scores, np.mean(r2_scores)

(array([0.84821541, 0.76792748, 0.8496905 , 0.82734008, 0.84881087]),
 np.float64(0.8283968696329669))

+ 교차검증 후 R2스코어는 약 0.83이다